# Exporing the data exported frm the Fitness app

Using CSV app

In [8]:
#import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
#from sklearn.linear_model import LinearRegression
#from sklearn.model_selection import train_test_split
#from sklearn.metrics import mean_squared_error
from pathlib import Path
from os import path, listdir
import re
from datetime import datetime,tzinfo
from dateutil import parser
import pytz

home = Path(path.abspath('..'))
user_home = Path('/Users/edmundlskoviak/')
data = home / 'book_data'
fitness_data = user_home / 'iCloud' / 'Data Sets' /'HealthAll_2024-12-345_14-58-25_SimpleHealthExportCSV'

In [35]:
# Test the fitness_data path
#json_files = [f for f in listdir(json_path) if path.isfile(path.join(json_path, f)) and f.split('.')[1] == 'json']
def convertDate(d : str) -> datetime:
    return parser.parse(d)

item = 'FlightsClimbed'
HKQuantity = re.compile(r'^HKQuantityTypeIdentifier'+item)
csv_files = [f for f in listdir(fitness_data) if path.isfile(path.join(fitness_data, f)) and f.split('.')[1] == 'csv' and HKQuantity.match(f)]
#dtypes = {"startDate": datetime64[ns, tzutc()], "endDate":datetime64[ns, tzutc()]}
converters={"startDate" : convertDate, "endDate" : convertDate}
columns = ['type','sourceVersion','productType','startDate','endDate','unit','value']
HKQuantity_df = pd.read_csv(fitness_data / csv_files[0], skiprows=0, header=1, usecols=columns, converters=converters)


HKQuantity_filtered_df = HKQuantity_df[HKQuantity_df['startDate'] < pd.Timestamp('2024-09-02',tz=pytz.UTC)]
print(f'{item}: {HKQuantity_filtered_df['value'].sum()}')

FlightsClimbed: 4.0


## Lets look at some activity

Filename starts with 'HKWorkoutActivityType<>'

Where <> in

* CrossTraining
* Cycling
* Eliptical
* FunctionalStrengthTraining
* Golf
* Hiking
* Other
* PaddleSports
* Running
* Snowboarding
* SnowSports
* Swimming
* TraditionalStrengthTraining (does not have HKElevationAscent)
* Walking (has HKElevationAscent), also has HKIndoorWorkout flag)
* WaterSports
* Yoga

for 9/1-

* 879/800 Cal (Move)
* 104/90 Min 
* 16/12 hrs

* 16728 steps
* 7.30 Mi
* 6 flights climbed

Outdoor walk 391 active cal

In [43]:
def convertDate(d : str) -> datetime:
    return parser.parse(d)

item = 'Yoga'
HKActivity = re.compile(r'^HKWorkoutActivityType'+item)
csv_files = [f for f in listdir(fitness_data) if path.isfile(path.join(fitness_data, f)) and f.split('.')[1] == 'csv' and HKActivity.match(f)]
#dtypes = {"startDate": datetime64[ns, tzutc()], "endDate":datetime64[ns, tzutc()]}
converters={"startDate" : convertDate, "endDate" : convertDate}
columns = ['type','sourceVersion','productType','startDate','endDate','activityType','duration','durationUnit','totalEnergyBurned','totalDistance',
           'totalSwimmingStrokeCount', 'totalFlightsClimbed','HKIndoorWorkout','HKWeatherHumidity','HKTimeZone','HKAverageMETs',
           #'HKElevationAscended',
           'HKWeatherTemperature']
HKActivity_df = pd.read_csv(fitness_data / csv_files[0], skiprows=0, header=1, usecols=columns, converters=converters)

HKActivity_filtered_df = HKActivity_df[HKActivity_df['startDate'] < pd.Timestamp('2024-11-16',tz=pytz.UTC)]
print(f'{item}: {HKActivity_filtered_df['totalEnergyBurned'].sum()}')


Yoga: 227.192 kcal


In [6]:
# Attempt to read a parquet file

parquet_file = data / 'sparkOutput' / 'activity'

recovered_df = pd.read_parquet(parquet_file)

# Fix the start date and end date
recovered_df['startDate'] = recovered_df.apply(lambda row: pd.Timestamp(row['startDate'], tz=pytz.UTC), axis=1)
recovered_df['endDate'] = recovered_df.apply(lambda row: pd.Timestamp(row['endDate'], tz=pytz.UTC), axis=1)


recovered_df[recovered_df['startDate'] < pd.Timestamp('2024-09-02',tz=pytz.UTC)]

,type,sourceName,sourceVersion,productType,device,startDate,endDate,activityType,duration,durationUnit,...,totalFlightsClimbed,HKElevationAscended,HKAverageMETs,HKWeatherHumidity,HKWeatherTemperature,HKTimeZone,HKIndoorWorkout,HKMaximumSpeed,HKAverageSpeed,HKElevationDescended
100,HKWorkoutTypeIdentifier,Edmund L’s Apple Watch,10.6.1,"Watch6,18","<<HKDevice: 0x3005b2710>, name:Apple Watch, ma...",2024-09-01 18:14:31+00:00,2024-09-01 20:12:41+00:00,Walking,4979.012531042099,sec,...,None,5715 cm,4.32501 kcal/hr·kg,4800 %,70.484 degF,America/Chicago,0,None,None,None


In [ ]:
# Read Activity File

activity_parquet_file = data / 'sparkUutput' / 'quantity'

quantity_df = pd.read_parquet(activity_parquet_file)





FileNotFoundError: [Errno 2] No such file or directory: '/Users/edmundlskoviak/Documents/repos/pyspark-playground/book_data/spark_output/quantity'